In [9]:
# Import necessary libraries
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import librosa
import wandb
from tqdm import tqdm

In [10]:
# Paths
DATA_ROOT = "../data/messy_mashup/genres_stems"
NOISE_DIR = "../data/messy_mashup/ESC-50-master/audio"
MODEL_DIR = "../models"
CACHE_DIR = "../working/efficientnet_cache"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

GENRES       = ["blues", "classical", "country", "disco", "hiphop",
                "jazz", "metal", "pop", "reggae", "rock"]
STEMS        = ["drums", "vocals", "bass", "other"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

In [11]:
# Configs
SR = 22050
DURATION = 10.0
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 256
SAMPLES_PER_GENRE = 200
VAL_SPLIT = 0.2
DATA_SEED = 67
TRAINING_SEED = 1234

GAIN_MIN = 0.5
GAIN_MAX = 1.5
SNR_MIN_DB = 5.0
SNR_MAX_DB = 20.0

# phase 1: head only
PHASE1_EPOCHS = 5
PHASE1_LR = 1e-3

# phase 2: unfreeze last block of EfficientNet
PHASE2_EPOCHS  = 15
PHASE2_LR_HEAD = 1e-4
PHASE2_LR_TRUNK = 1e-5

BATCH_SIZE = 32
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

# EfficientNet-B0 expects 3-channel 224x224 input — ImageNet normalisation
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

spec_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)
torch.cuda.manual_seed_all(1234)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [12]:
# Helper functions (from eda_and_process notebook)
def _pad_or_trim(y, target_len):
    if len(y) >= target_len:
        return y[:target_len]
    return np.pad(y, (0, target_len - len(y)))


def make_augmented_mix(genre, data_root, sr, duration, rng):
    """
    Build one training sample via cross-song stem mixing.
    Each of the 4 stems is drawn from a independently chosen song of the
    same genre, simulating the cross-song recombination in the test mashups.
    Per-stem random gain, time-stretch, and variable-SNR ESC-50 noise are
    applied before peak normalisation.
    """
    target_len = int(sr * duration)
    genre_dir  = os.path.join(data_root, genre)
    songs      = sorted(os.listdir(genre_dir))
    mix        = np.zeros(target_len, dtype=np.float32)

    for stem in STEMS:
        song = rng.choice(songs)
        path = os.path.join(genre_dir, song, f"{stem}.wav")
        try:
            y, _ = librosa.load(path, sr=sr)
            # random crop if stem is longer than target
            if len(y) > target_len:
                start = rng.randint(0, len(y) - target_len)
                y     = y[start: start + target_len]
            else:
                y = _pad_or_trim(y, target_len)
            gain  = rng.uniform(GAIN_MIN, GAIN_MAX)
            mix  += y * gain
        except Exception:
            pass

    # time stretch removed, too slow for on-the-fly augmentation
    # and causes spectrogram time dimension instability
    # rate = rng.uniform(STRETCH_MIN, STRETCH_MAX)
    # try:
    #     mix = librosa.effects.time_stretch(mix, rate=rate)
    # except Exception:
    #     pass
    # mix = _pad_or_trim(mix, target_len)

    # add ESC-50 noise at a random SNR
    noise_files = [f for f in os.listdir(NOISE_DIR) if f.endswith(".wav")]
    try:
        noise, _ = librosa.load(
            os.path.join(NOISE_DIR, rng.choice(noise_files)), sr=sr
        )
        noise       = _pad_or_trim(noise, target_len)
        snr_db      = rng.uniform(SNR_MIN_DB, SNR_MAX_DB)
        sig_rms     = np.sqrt(np.mean(mix   ** 2)) + 1e-9
        noise_rms   = np.sqrt(np.mean(noise ** 2)) + 1e-9
        target_rms  = sig_rms / (10 ** (snr_db / 20.0))
        mix         = mix + noise * (target_rms / noise_rms)
    except Exception:
        pass

    peak = np.max(np.abs(mix))
    if peak > 0:
        mix /= (peak + 1e-9)
    return mix

In [13]:
# Audio to mel-spectrogram function 
def audio_to_melspec_3ch(y, sr, n_mels, n_fft, hop_length):
    """
    Converts a waveform to a 3-channel mel-spectrogram tensor suitable
    for EfficientNet. The single-channel spectrogram is replicated across
    3 channels to match ImageNet pretraining input format.
    """
    S    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
                                          n_fft=n_fft, hop_length=hop_length)
    S_db = librosa.power_to_db(S, ref=np.max).astype(np.float32)
    S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)
    spec = torch.tensor(S_db).unsqueeze(0).repeat(3, 1, 1)
    return spec_transform(spec)

In [14]:
class PrecomputedEfficientNetDataset(Dataset):
    def __init__(self, cache_dir, split):
        self.split_dir = os.path.join(cache_dir, split)
        self.files     = sorted(os.listdir(self.split_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fpath = os.path.join(self.split_dir, self.files[idx])
        data  = torch.load(fpath, weights_only=True)
        return data["input_values"], data["label"]

In [15]:
# Function to precompute efficientnet features
def precompute_efficientnet_features(cache_dir, genres, sr, duration,
                                     n_mels, n_fft, hop_length,
                                     samples_per_genre, val_split, seed):
    all_items = [(g, i) for g in genres for i in range(samples_per_genre)]
    train_items, val_items = train_test_split(
        all_items, test_size=val_split, random_state=seed,
        stratify=[g for g, _ in all_items]
    )

    for split, items in [("train", train_items), ("val", val_items)]:
        split_dir = os.path.join(cache_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        for pos, (genre, idx) in enumerate(tqdm(items, desc=f"Precomputing [{split}]")):
            out_path = os.path.join(split_dir, f"{split}_{pos:05d}.pt")
            if os.path.exists(out_path):
                continue
            rng = random.Random(seed + pos)
            mix = make_augmented_mix(genre, DATA_ROOT, sr, duration, rng)
            spec = audio_to_melspec_3ch(mix, sr, n_mels, n_fft, hop_length)
            torch.save({
                "input_values": spec,
                "label":        GENRE_TO_IDX[genre]
            }, out_path)

    print("Precomputation complete.")

In [16]:
# Precompute efficientnet features if cache does not exist
if not os.path.exists(os.path.join(CACHE_DIR, "train")) or \
        len(os.listdir(os.path.join(CACHE_DIR, "train"))) == 0:
    precompute_efficientnet_features(
        cache_dir=CACHE_DIR, genres=GENRES, sr=SR, duration=DURATION,
        n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH,
        samples_per_genre=SAMPLES_PER_GENRE, val_split=VAL_SPLIT, seed=DATA_SEED
    )
else:
    print("EfficientNet feature cache already exists, skipping precomputation.")

Precomputing [val]: 100%|██████████| 400/400 [01:15<00:00,  5.31it/s]

Precomputation complete.


In [17]:
# Create dataloaders
train_dataset = PrecomputedEfficientNetDataset(CACHE_DIR, "train")
val_dataset   = PrecomputedEfficientNetDataset(CACHE_DIR, "val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

Train samples: 1600 | Val samples: 400


In [18]:
# Define model
backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

# replace the classifier head for 10-class output
backbone.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(backbone.classifier[1].in_features, len(GENRES))
)

# freeze everything except the new classifier head for phase 1
for name, param in backbone.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

model     = backbone.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1 trainable parameters: {trainable:,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Atul/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 39.3MB/s]


Phase 1 trainable parameters: 12,810


In [19]:
# Run epoch function
def run_epoch(model, loader, criterion, optimizer, training, device, desc=""):
    model.train() if training else model.eval()
    total_loss, preds, labels_all = 0.0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for inputs, labels in tqdm(loader, desc=desc):
            inputs, labels = inputs.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            logits = model(inputs)
            loss   = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss  += loss.item() * inputs.size(0)
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1       = f1_score(labels_all, preds, average="macro")
    acc      = accuracy_score(labels_all, preds)
    return avg_loss, f1, acc


criterion = nn.CrossEntropyLoss()

In [20]:
# Initialize wandb
wandb.init(
    project="21f3002298-dl-genai-project",
    name="efficientnet-src-phase1",
    config={
        "model":          "EfficientNet-B0 (ImageNet pretrained)",
        "phase1_epochs":  PHASE1_EPOCHS,
        "phase1_lr":      PHASE1_LR,
        "phase2_epochs":  PHASE2_EPOCHS,
        "phase2_lr_head": PHASE2_LR_HEAD,
        "phase2_lr_trunk": PHASE2_LR_TRUNK,
        "batch_size":     BATCH_SIZE,
        "weight_decay":   WEIGHT_DECAY,
        "sr":             SR,
        "duration":       DURATION,
        "n_mels":         N_MELS,
        "hop_length":     HOP_LENGTH,
        "augmentation":   "cross-song mixing, per-stem gain, variable SNR",
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Atul\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [21]:
best_val_f1 = 0.0

print("Phase 1: head warmup")
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_LR, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

for epoch in range(PHASE1_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch(
        model, train_loader, criterion, optimizer, training=True,
        device=DEVICE, desc=f"[P1] Epoch {epoch+1}/{PHASE1_EPOCHS} [Train]"
    )
    val_loss, val_f1, val_acc = run_epoch(
        model, val_loader, criterion, optimizer, training=False,
        device=DEVICE, desc=f"[P1] Epoch {epoch+1}/{PHASE1_EPOCHS} [Val]"
    )
    scheduler.step()

    wandb.log({
        "epoch": epoch + 1, "phase": 1,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss":   val_loss,   "val_f1":   val_f1,   "val_acc":   val_acc,
        "lr": scheduler.get_last_lr()[0],
    })
    print(f"[P1] Epoch {epoch+1}/{PHASE1_EPOCHS}: "
          f"train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(),
                   os.path.join(MODEL_DIR, "efficientnet_src_phase1_best.pth"))
        print(f"  Saved best model (val_f1={val_f1:.4f})")

Phase 1: head warmup


[P1] Epoch 1/5 [Val]: 100%|██████████| 13/13 [00:01<00:00,  7.71it/s]


[P1] Epoch 1/5: train_f1=0.3295 | val_f1=0.4369
  Saved best model (val_f1=0.4369)


[P1] Epoch 2/5 [Val]: 100%|██████████| 13/13 [00:01<00:00,  9.70it/s]


[P1] Epoch 2/5: train_f1=0.5523 | val_f1=0.5775
  Saved best model (val_f1=0.5775)


[P1] Epoch 3/5 [Val]: 100%|██████████| 13/13 [00:01<00:00,  9.92it/s]


[P1] Epoch 3/5: train_f1=0.5987 | val_f1=0.5978
  Saved best model (val_f1=0.5978)


[P1] Epoch 4/5 [Val]: 100%|██████████| 13/13 [00:01<00:00, 10.12it/s]


[P1] Epoch 4/5: train_f1=0.6323 | val_f1=0.5953


[P1] Epoch 5/5 [Val]: 100%|██████████| 13/13 [00:01<00:00,  9.67it/s]


[P1] Epoch 5/5: train_f1=0.6541 | val_f1=0.6003
  Saved best model (val_f1=0.6003)


In [22]:
print(f"\nPhase 2: unfreezing last EfficientNet block")

for param in model.parameters():
    param.requires_grad = False

# unfreeze the last MBConv block (features[-1]) and the classifier head
for param in model.features[-1].parameters():
    param.requires_grad = True
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2 trainable parameters: {trainable:,}")

head_params  = list(model.classifier.parameters())
trunk_params = list(model.features[-1].parameters())

optimizer = optim.AdamW([
    {"params": head_params,  "lr": PHASE2_LR_HEAD},
    {"params": trunk_params, "lr": PHASE2_LR_TRUNK},
], weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

best_val_f1_p2 = 0.0
wandb.run.name = "efficientnet-src-phase2"

for epoch in range(PHASE2_EPOCHS):
    train_loss, train_f1, train_acc = run_epoch(
        model, train_loader, criterion, optimizer, training=True,
        device=DEVICE, desc=f"[P2] Epoch {epoch+1}/{PHASE2_EPOCHS} [Train]"
    )
    val_loss, val_f1, val_acc = run_epoch(
        model, val_loader, criterion, optimizer, training=False,
        device=DEVICE, desc=f"[P2] Epoch {epoch+1}/{PHASE2_EPOCHS} [Val]"
    )
    scheduler.step()

    wandb.log({
        "epoch": PHASE1_EPOCHS + epoch + 1, "phase": 2,
        "train_loss": train_loss, "train_f1": train_f1, "train_acc": train_acc,
        "val_loss":   val_loss,   "val_f1":   val_f1,   "val_acc":   val_acc,
        "lr_head":  optimizer.param_groups[0]["lr"],
        "lr_trunk": optimizer.param_groups[1]["lr"],
    })
    print(f"[P2] Epoch {epoch+1}/{PHASE2_EPOCHS}: "
          f"train_f1={train_f1:.4f} | val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1_p2:
        best_val_f1_p2 = val_f1
        torch.save(model.state_dict(),
                   os.path.join(MODEL_DIR, "efficientnet_src_phase2_best.pth"))
        print(f"  Saved best model (val_f1={val_f1:.4f})")


Phase 2: unfreezing last EfficientNet block
Phase 2 trainable parameters: 424,970


[P2] Epoch 1/15 [Val]: 100%|██████████| 13/13 [00:01<00:00, 10.09it/s]


[P2] Epoch 1/15: train_f1=0.6380 | val_f1=0.6010
  Saved best model (val_f1=0.6010)


[P2] Epoch 2/15 [Val]: 100%|██████████| 13/13 [00:01<00:00, 10.12it/s]


[P2] Epoch 2/15: train_f1=0.6575 | val_f1=0.6020
  Saved best model (val_f1=0.6020)


[P2] Epoch 3/15 [Val]: 100%|██████████| 13/13 [00:01<00:00,  9.50it/s]


... (21 outputs left out)

[P2] Epoch 14/15 [Val]: 100%|██████████| 13/13 [00:01<00:00,  8.68it/s]


[P2] Epoch 14/15: train_f1=0.6838 | val_f1=0.6079


[P2] Epoch 15/15 [Val]: 100%|██████████| 13/13 [00:02<00:00,  5.25it/s]

[P2] Epoch 15/15: train_f1=0.6936 | val_f1=0.6098


In [23]:
wandb.finish()
print(f"Training complete. Best phase 1 val F1: {best_val_f1:.4f}")
print(f"Training complete. Best phase 2 val F1: {best_val_f1_p2:.4f}")

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,█▆▄▂▁
lr_head,██▇▇▆▆▅▄▃▃▂▂▁▁▁
lr_trunk,██▇▇▆▆▅▄▃▃▂▂▁▁▁
phase,▁▁▁▁▁███████████████
train_acc,▁▅▆▇▇▇▇▇▇▇██████████
train_f1,▁▅▆▇▇▇▇▇▇▇██████████
train_loss,█▅▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▆▇▇█████▇▇█████████
val_f1,▁▇▇▇██████▇█████████
+1,...


Training complete. Best phase 1 val F1: 0.6003
Training complete. Best phase 2 val F1: 0.6119
